# Neural CFR+ reference on the first approximate-scale game

This notebook trains neural CFR+ on `r5_s4_h3_hp2ptq_ss`, a 30-claim game for which dense exact exploitability is no longer operationally practical.

The workflow is deliberately split:

1. run 45 uninterrupted measured minutes of CFR+ and save lightweight average-policy snapshots at 5, 10, 20, 30 and 45 minutes;
2. after training, evaluate each snapshot serially with three action-conditioned fitted-return responder seeds, five measured responder-training minutes per seed;
3. plot the discovered exploitability estimate and conservative Monte Carlo lower bound.

Approximate responder evaluation is post hoc, so it cannot slow or perturb the CFR+ training trajectory.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.neural_evaluators import (
    PolicySnapshotEvaluator,
    ScheduledEvaluation,
)
from liars_poker.training.br_runs import run_best_responder
from liars_poker.training.neural_runs import run_neural_cfr_plus

assert torch.cuda.is_available(), 'This reference run requires CUDA.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=5,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'Quads'),
    suit_symmetry=True,
)

TRAINING_MINUTES = 45
SNAPSHOT_MINUTES = (5, 10, 20, 30, 45)
CHECKPOINT_EVERY_MINUTES = 15
SAVE_RESUME_CHECKPOINT = True
TRAVERSALS_PER_PLAYER = 1024
SEED = 7
RUN_REFERENCE = True

TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = (
    REPO_ROOT / 'artifacts' / 'cfr_plus_approx_reference_runs'
    / f'{spec.to_short_str()}___{run_id}'
)
print('claims:', len(rules_for_spec(spec).claims))
print('run directory:', RUN_DIR)

In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]


def mean_validation(metrics, section, metric):
    values = [row[metric] for row in metrics[section] if row.get('records', 0)]
    return float(np.mean(values)) if values else np.nan


snapshot_schedule = ScheduledEvaluation(
    name='policy_snapshots',
    evaluator=PolicySnapshotEvaluator(),
    at_minutes=SNAPSHOT_MINUTES,
    run_at_end=True,
)

In [ ]:
reference_result = None

if RUN_REFERENCE:
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise FileExistsError(
            f'{RUN_DIR} already contains data. Rerun the configuration cell '
            'for a new run ID, or resume explicitly with the training API.'
        )

    reference_result = run_neural_cfr_plus(
        spec,
        minutes=TRAINING_MINUTES,
        traversals_per_player=TRAVERSALS_PER_PLAYER,
        trainer_kwargs=TRAINER_KWARGS,
        evaluations=[snapshot_schedule],
        checkpoint_every_minutes=(
            CHECKPOINT_EVERY_MINUTES if SAVE_RESUME_CHECKPOINT else None
        ),
        run_dir=RUN_DIR,
        save_checkpoint=SAVE_RESUME_CHECKPOINT,
        wait_for_evaluations=True,
        debug=True,
    )

    validation = reference_result.trainer.validation_metrics()
    diagnostics = pd.DataFrame([{
        'iterations': reference_result.trainer.iteration,
        'measured training min': reference_result.measured_training_s / 60.0,
        'wall min': reference_result.wall_s / 60.0,
        'final regret MSE': mean_validation(validation, 'regret', 'mse'),
        'final regret strategy TV': mean_validation(validation, 'regret', 'strategy_tv'),
        'final average-network TV': mean_validation(validation, 'strategy', 'strategy_tv'),
    }])
    diagnostics.to_csv(RUN_DIR / 'final_diagnostics.csv', index=False)
    display(diagnostics.style.format(precision=6))
    print('completed:', reference_result.run_dir)

    del reference_result
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
training = pd.DataFrame(read_jsonl(RUN_DIR / 'training.jsonl'))
snapshot_records = pd.DataFrame(read_jsonl(RUN_DIR / 'evaluations.jsonl'))
snapshot_records = snapshot_records[
    snapshot_records['evaluator'] == 'policy_snapshots'
].sort_values('measured_training_s').drop_duplicates('iteration', keep='last')
snapshot_records['training_min'] = snapshot_records['measured_training_s'] / 60.0

timing = pd.json_normalize(training['timing'])
training_summary = pd.DataFrame([{
    'iterations': int(training['iteration'].iloc[-1]),
    'measured training min': training['measured_training_s'].iloc[-1] / 60.0,
    'iterations / min': (
        training['iteration'].iloc[-1]
        / (training['measured_training_s'].iloc[-1] / 60.0)
    ),
    'mean traversal s': timing['traversal_s'].mean(),
    'mean regret fit s': timing['regret_training_s'].mean(),
    'mean strategy fit s': timing['strategy_training_s'].mean(),
}])
display(training_summary.style.format(precision=6))
display(snapshot_records[[
    'iteration', 'training_min', 'snapshot_s', 'policy_dir'
]].style.format(precision=4))

## Serial approximate best-response evaluation

Each following cell evaluates one frozen average-policy snapshot. It runs seeds 7, 17 and 27 sequentially, five measured responder-training minutes each. Monte Carlo evaluation occurs every responder-training minute and does not count against that five-minute budget.

Every seed writes its own logs and final policy immediately. Rerunning a completed snapshot cell loads those records instead of retraining.

In [ ]:
BR_SEEDS = (7, 17, 27)
BR_MINUTES_PER_SEED = 5
BR_EVALUATE_EVERY_MINUTES = 1
BR_EVAL_EPISODES_PER_ROLE = 200_000
BR_EPISODES_PER_ROLE = 4096
BR_ROLLOUT_BATCH_SIZE = 1024

BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': device,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
}


def snapshot_for(target_minutes):
    distance = (snapshot_records['training_min'] - float(target_minutes)).abs()
    row = snapshot_records.loc[distance.idxmin()]
    if abs(float(row['training_min']) - float(target_minutes)) > 2.0:
        raise RuntimeError(f'No snapshot close to {target_minutes} minutes.')
    policy_dir = Path(row['policy_dir'])
    if not policy_dir.exists():
        candidates = list(
            (RUN_DIR / 'evaluations' / 'policy_snapshots' / 'snapshots').glob(
                f'*iter_{int(row["iteration"]):08d}_*'
            )
        )
        if len(candidates) != 1:
            raise FileNotFoundError(f'Cannot resolve snapshot policy: {policy_dir}')
        policy_dir = candidates[0]
    return row, policy_dir


def evaluate_snapshot(target_minutes):
    snapshot, policy_dir = snapshot_for(target_minutes)
    output_root = RUN_DIR / 'approx_br' / f'train_{int(target_minutes):03d}m'
    rows = []

    for seed in BR_SEEDS:
        seed_dir = output_root / f'seed_{seed}'
        metrics_path = seed_dir / 'metrics.json'
        evaluations_path = seed_dir / 'evaluations.jsonl'

        if metrics_path.exists() and evaluations_path.exists():
            print(f'loaded train={target_minutes}m seed={seed}')
        else:
            if seed_dir.exists() and any(seed_dir.iterdir()):
                raise FileExistsError(
                    f'Incomplete responder directory: {seed_dir}. '
                    'Inspect or remove it before rerunning this seed.'
                )
            kwargs = dict(BR_TRAINER_KWARGS)
            kwargs['seed'] = seed
            print(f'\n=== snapshot={target_minutes}m seed={seed} ===')
            result = run_best_responder(
                policy_dir,
                method='action_conditioned_fitted_return',
                minutes=BR_MINUTES_PER_SEED,
                trainer_kwargs=kwargs,
                episodes_per_role=BR_EPISODES_PER_ROLE,
                rollout_batch_size=BR_ROLLOUT_BATCH_SIZE,
                evaluate_every_minutes=BR_EVALUATE_EVERY_MINUTES,
                eval_episodes_per_role=BR_EVAL_EPISODES_PER_ROLE,
                exact_evaluation=False,
                run_dir=seed_dir,
                debug=True,
            )
            del result
            gc.collect()
            torch.cuda.empty_cache()

        seed_rows = pd.DataFrame(read_jsonl(evaluations_path))
        seed_rows['snapshot_target_min'] = float(target_minutes)
        seed_rows['snapshot_training_min'] = float(snapshot['training_min'])
        seed_rows['snapshot_iteration'] = int(snapshot['iteration'])
        seed_rows['seed'] = int(seed)
        seed_rows['run_dir'] = str(seed_dir)
        rows.append(seed_rows)

    frame = pd.concat(rows, ignore_index=True)
    frame.to_csv(output_root / 'combined_evaluations.csv', index=False)
    return frame


def load_all_br_results():
    files = sorted((RUN_DIR / 'approx_br').glob('train_*m/combined_evaluations.csv'))
    if not files:
        raise RuntimeError('Run at least one snapshot evaluation first.')
    return pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

In [ ]:
br_005m = evaluate_snapshot(5)

In [ ]:
br_010m = evaluate_snapshot(10)

In [ ]:
br_020m = evaluate_snapshot(20)

In [ ]:
br_030m = evaluate_snapshot(30)

In [ ]:
br_045m = evaluate_snapshot(45)

In [ ]:
br_results = load_all_br_results()

seed_rows = []
for (target_min, seed), frame in br_results.groupby(
    ['snapshot_target_min', 'seed'], sort=True
):
    seed_rows.append({
        'snapshot target min': target_min,
        'snapshot training min': frame['snapshot_training_min'].iloc[0],
        'snapshot iteration': int(frame['snapshot_iteration'].iloc[0]),
        'seed': int(seed),
        'best p_first': frame['p_first'].max(),
        'best p_second': frame['p_second'].max(),
        'best p_first LCB': frame['p_first_lcb'].max(),
        'best p_second LCB': frame['p_second_lcb'].max(),
        'best discovered estimate': frame['p_first'].max() + frame['p_second'].max() - 1.0,
        'conservative lower bound': frame['p_first_lcb'].max() + frame['p_second_lcb'].max() - 1.0,
    })
seed_summary = pd.DataFrame(seed_rows)

snapshot_rows = []
for target_min, frame in seed_summary.groupby('snapshot target min', sort=True):
    snapshot_rows.append({
        'snapshot target min': target_min,
        'snapshot training min': frame['snapshot training min'].iloc[0],
        'snapshot iteration': int(frame['snapshot iteration'].iloc[0]),
        'best discovered estimate': frame['best p_first'].max() + frame['best p_second'].max() - 1.0,
        'conservative lower bound': frame['best p_first LCB'].max() + frame['best p_second LCB'].max() - 1.0,
        'mean seed estimate': frame['best discovered estimate'].mean(),
        'seed estimate std': frame['best discovered estimate'].std(),
    })
snapshot_summary = pd.DataFrame(snapshot_rows).set_index('snapshot target min')
display(snapshot_summary.style.format(precision=6))
display(seed_summary.set_index(['snapshot target min', 'seed']).style.format(precision=6))

snapshot_summary.to_csv(RUN_DIR / 'approx_br_snapshot_summary.csv')
seed_summary.to_csv(RUN_DIR / 'approx_br_seed_summary.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2))

x = snapshot_summary['snapshot training min']
axes[0].plot(
    x,
    snapshot_summary['best discovered estimate'],
    marker='o',
    label='best discovered estimate',
)
axes[0].plot(
    x,
    snapshot_summary['conservative lower bound'],
    marker='o',
    label='conservative MC lower bound',
)
axes[0].set(
    title='Approximate exploitability over CFR+ training',
    xlabel='Measured CFR+ training minutes',
    ylabel='Discovered exploitability',
)
axes[0].grid(alpha=0.3)
axes[0].legend()

for seed, frame in seed_summary.groupby('seed'):
    axes[1].plot(
        frame['snapshot training min'],
        frame['best discovered estimate'],
        marker='o',
        label=f'seed {seed}',
    )
axes[1].set(
    title='Responder-seed agreement',
    xlabel='Measured CFR+ training minutes',
    ylabel='Best discovered estimate',
)
axes[1].grid(alpha=0.3)
axes[1].legend()
fig.tight_layout()

In [ ]:
final_target = max(br_results['snapshot_target_min'])
final_results = br_results[br_results['snapshot_target_min'] == final_target]

fig, ax = plt.subplots(figsize=(8.5, 5.2))
for seed, frame in final_results.groupby('seed'):
    frame = frame.sort_values('measured_training_s').copy()
    frame['best p_first'] = frame['p_first'].cummax()
    frame['best p_second'] = frame['p_second'].cummax()
    frame['best discovered'] = frame['best p_first'] + frame['best p_second'] - 1.0
    ax.plot(
        frame['measured_training_s'] / 60.0,
        frame['best discovered'],
        marker='o',
        label=f'seed {seed}',
    )
ax.set(
    title=f'Responder compute curve for the {final_target:.0f}-minute CFR+ policy',
    xlabel='Measured responder-training minutes',
    ylabel='Best discovered exploitability estimate',
)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

## Interpretation

The plotted quantities are lower-bound diagnostics, not exact exploitability. Improvement across CFR+ snapshots is credible when discovered exploitability falls consistently while independent responder seeds agree and each final responder-compute curve is close to plateauing. A rising or highly dispersed curve indicates that evaluator strength, rather than CFR+ policy quality, is limiting the measurement.

The reported Wilson lower bound is conservative for each individual held-out evaluation. Because the notebook selects the best checkpoint and seed afterward, it is not a simultaneous 95% confidence guarantee across the whole search.

In [ ]:
# Optional cleanup after verifying the policy, snapshots, and responder results.
DELETE_RESUME_CHECKPOINT = False

if DELETE_RESUME_CHECKPOINT:
    checkpoint = RUN_DIR / 'latest_checkpoint.pt'
    if checkpoint.exists():
        checkpoint.unlink()
        print('removed:', checkpoint)
    else:
        print('no checkpoint present')